# Epidata Summary Statistics

Summary of BEAST2 phylogenetic simulation outputs. Each dataset contains multiple outbreaks with:
- `*_beast2.trees` — phylogenetic tree (1 per outbreak)
- `*_nf.csv` — per-location metrics (n rows per outbreak, where n = number of locations)
- `*_parameter.csv` — simulation parameters including migration rates

**Statistics (Min / Mean / Max / Std / N):**
1. Tree Size — tips per tree (1 per outbreak)
2. R0 — reproduction number (n per outbreak)
3. Recovery Rate (n per outbreak)
4. Source Sink Score (n per outbreak)
5. Ancestral State Distribution + match rate with earliest tip
6. Migration Rate — n×(n-1) per outbreak
7. Sampling Rate & Percentage (1 per outbreak each)

*n = number of locations/nodes (e.g., 16)*

In [1]:
import re, csv
import numpy as np
from pathlib import Path
from collections import Counter

# Regex patterns for BEAST2 tree parsing
TIP_RE = re.compile(r'\[&type="I\{(\d+)\}",samp="sample",time=([\d.]+)\]')
TREE_RE = re.compile(r'tree STATE_\d+ = (.+?)\n')

DATASETS = [
    '/Users/lukelyu/Desktop/epidata/10000_1_0.0006_0.0007/',
    '/Users/lukelyu/Desktop/epidata/10000_1_0.0006_0.0015/',
    '/Users/lukelyu/Desktop/epidata/10000_1_0.0006_0.0023/'
]

def print_stats(name, arr):
    """Print Min/Mean/Max/Std/N statistics for an array."""
    if len(arr) == 0:
        print(f'{name:40s}  {"N/A":>10s}  {"N/A":>10s}  {"N/A":>10s}  {"N/A":>10s}  {0:>8d}')
    else:
        print(f'{name:40s}  {arr.min():10.4f}  {arr.mean():10.4f}  {arr.max():10.4f}  {arr.std():10.4f}  {len(arr):8d}')

def print_header():
    print(f'{"Dataset":40s}  {"Min":>10s}  {"Mean":>10s}  {"Max":>10s}  {"Std":>10s}  {"N":>8s}')
    print('-' * 95)

In [2]:
# 1. Tree Size (1 per outbreak)
def get_tree_sizes(ds):
    """Count sampled tips per tree."""
    sizes = []
    for tf in Path(ds).glob('*_beast2.trees'):
        tree_str = TREE_RE.search(tf.read_text()).group(1)
        sizes.append(len(TIP_RE.findall(tree_str)))
    return np.array(sizes, dtype=float)

print("=== 1. Tree Size ===\n")
print_header()
for ds in DATASETS:
    print_stats(Path(ds).name, get_tree_sizes(ds))

=== 1. Tree Size ===

Dataset                                          Min        Mean         Max         Std         N
-----------------------------------------------------------------------------------------------
10000_1_0.0006_0.0007                       356.0000    965.9836   2159.0000    316.1571     10000
10000_1_0.0006_0.0015                       337.0000    943.7717   2379.0000    320.0052     10000
10000_1_0.0006_0.0023                       366.0000    937.7992   2271.0000    321.8102     10000


In [3]:
# 2-4. Node Features from *_nf.csv (n per outbreak)
def collect_nf_column(ds, col):
    """Collect column values from all *_nf.csv files."""
    vals = []
    for nf in Path(ds).glob('*_nf.csv'):
        with open(nf) as f:
            vals.extend(float(row[col]) for row in csv.DictReader(f))
    return np.array(vals)

for title, col in [("2. R0", "R0"), ("3. Recovery Rate", "Recovery_Rate"), ("4. Source Sink Score", "Source_Sink_Score")]:
    print(f"=== {title} ===\n")
    print_header()
    for ds in DATASETS:
        print_stats(Path(ds).name, collect_nf_column(ds, col))
    print()

=== 2. R0 ===

Dataset                                          Min        Mean         Max         Std         N
-----------------------------------------------------------------------------------------------
10000_1_0.0006_0.0007                         2.0112      5.0222      7.9921      1.2836    160000
10000_1_0.0006_0.0015                         2.0063      5.0077      7.9971      1.2913    160000
10000_1_0.0006_0.0023                         2.0087      5.0025      7.9966      1.2823    160000

=== 3. Recovery Rate ===

Dataset                                          Min        Mean         Max         Std         N
-----------------------------------------------------------------------------------------------
10000_1_0.0006_0.0007                         0.0100      0.0287      0.0500      0.0087    160000
10000_1_0.0006_0.0015                         0.0100      0.0294      0.0500      0.0090    160000
10000_1_0.0006_0.0023                         0.0101      0.0296      0.0

In [4]:
# 5. Ancestral State Distribution (1 per outbreak)
def get_ancestral_stats(ds):
    """Get ancestral state distribution and match rate with earliest tip."""
    ds = Path(ds)
    locs, matches, total = [], 0, 0
    for tf in ds.glob('*_beast2.trees'):
        nf = ds / tf.name.replace('_beast2.trees', '_nf.csv')
        if not nf.exists():
            continue
        # Get ancestral state location
        with open(nf) as f:
            anc = next((int(r['Location']) for r in csv.DictReader(f) if r['Ancestral_State'] == '1'), None)
        # Get earliest tip location
        tree_str = TREE_RE.search(tf.read_text()).group(1)
        earliest = int(min(TIP_RE.findall(tree_str), key=lambda x: float(x[1]))[0])
        locs.append(anc)
        matches += (earliest == anc)
        total += 1
    return Counter(locs), max(locs) + 1, matches, total

print("=== 5. Ancestral State Distribution ===\n")
for ds in DATASETS:
    counts, n_locs, matches, total = get_ancestral_stats(ds)
    print(f"{Path(ds).name}")
    print(f"  Loc:  " + "".join(f"{i:>6d}" for i in range(n_locs)))
    print(f"  N:    " + "".join(f"{counts.get(i,0):>6d}" for i in range(n_locs)))
    print(f"  %:    " + "".join(f"{counts.get(i,0)/total*100:>5.1f}%" for i in range(n_locs)))
    print(f"  Match (ancestral == earliest tip): {matches}/{total} ({matches/total:.1%})\n")

=== 5. Ancestral State Distribution ===

10000_1_0.0006_0.0007
  Loc:       0     1     2     3     4     5     6     7     8     9    10    11    12    13    14    15
  N:       634   613   614   588   597   615   684   616   665   643   606   591   651   651   597   635
  %:      6.3%  6.1%  6.1%  5.9%  6.0%  6.2%  6.8%  6.2%  6.7%  6.4%  6.1%  5.9%  6.5%  6.5%  6.0%  6.3%
  Match (ancestral == earliest tip): 7744/10000 (77.4%)

10000_1_0.0006_0.0015
  Loc:       0     1     2     3     4     5     6     7     8     9    10    11    12    13    14    15
  N:       645   600   615   593   618   618   632   630   624   692   610   644   616   679   607   577
  %:      6.5%  6.0%  6.2%  5.9%  6.2%  6.2%  6.3%  6.3%  6.2%  6.9%  6.1%  6.4%  6.2%  6.8%  6.1%  5.8%
  Match (ancestral == earliest tip): 6397/10000 (64.0%)

10000_1_0.0006_0.0023
  Loc:       0     1     2     3     4     5     6     7     8     9    10    11    12    13    14    15
  N:       608   637   628   609   631   585

In [5]:
# 6. Migration Rate (n*(n-1) per outbreak)
def get_migration_rates(ds):
    """Collect all migration rates from *_parameter.csv."""
    rates = []
    for pf in Path(ds).glob('*_parameter.csv'):
        with open(pf) as f:
            for row in csv.DictReader(f):
                rates.extend(float(row[c]) for c in row if c.startswith('migration_loc_'))
    return np.array(rates)

print("=== 6. Migration Rate ===\n")
print_header()
for ds in DATASETS:
    print_stats(Path(ds).name, get_migration_rates(ds))

=== 6. Migration Rate ===

Dataset                                          Min        Mean         Max         Std         N
-----------------------------------------------------------------------------------------------
10000_1_0.0006_0.0007                         0.0001      0.0004      0.0007      0.0002   2400000
10000_1_0.0006_0.0015                         0.0001      0.0008      0.0015      0.0004   2400000
10000_1_0.0006_0.0023                         0.0001      0.0012      0.0023      0.0006   2400000


In [6]:
# 7. Sampling Rate & Percentage (1 per outbreak each)
def get_sampling_rates(ds):
    """Get sample_rate from *_parameter.csv."""
    rates = []
    for pf in Path(ds).glob('*_parameter.csv'):
        with open(pf) as f:
            rates.extend(float(row['sample_rate']) for row in csv.DictReader(f))
    return np.array(rates)

def get_sampling_pcts(ds):
    """Compute tree_size / sum(Accumulated_Infections) per outbreak."""
    ds = Path(ds)
    pcts = []
    for tf in ds.glob('*_beast2.trees'):
        nf = ds / tf.name.replace('_beast2.trees', '_nf.csv')
        if not nf.exists():
            continue
        tree_str = TREE_RE.search(tf.read_text()).group(1)
        n_tips = len(TIP_RE.findall(tree_str))
        with open(nf) as f:
            total_inf = sum(int(row['Accumulated_Infections']) for row in csv.DictReader(f))
        if total_inf > 0:
            pcts.append(n_tips / total_inf)
    return np.array(pcts)

print("=== 7a. Sampling Rate (from *_parameter.csv) ===\n")
print_header()
for ds in DATASETS:
    print_stats(Path(ds).name, get_sampling_rates(ds))

print("\n=== 7b. Sampling Percentage (tree_size / accumulated_infections) ===\n")
print_header()
for ds in DATASETS:
    print_stats(Path(ds).name, get_sampling_pcts(ds))

=== 7a. Sampling Rate (from *_parameter.csv) ===

Dataset                                          Min        Mean         Max         Std         N
-----------------------------------------------------------------------------------------------
10000_1_0.0006_0.0007                         0.0006      0.0006      0.0006      0.0000     10000
10000_1_0.0006_0.0015                         0.0006      0.0006      0.0006      0.0000     10000
10000_1_0.0006_0.0023                         0.0006      0.0006      0.0006      0.0000     10000

=== 7b. Sampling Percentage (tree_size / accumulated_infections) ===

Dataset                                          Min        Mean         Max         Std         N
-----------------------------------------------------------------------------------------------
10000_1_0.0006_0.0007                         0.0086      0.0180      0.0310      0.0045     10000
10000_1_0.0006_0.0015                         0.0076      0.0149      0.0250      0.0033     